# Try ReFactX

In order to avoid to ingest the full 800-million-facts tree, this notebook uses a small in-memory prefix tree of 31,584 facts about famous artists and directors.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import time
from transformers.generation.logits_process import LogitsProcessorList
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer

import refactx
import os
from dotenv import load_dotenv

from peft import PeftModel

/opt/conda/envs/trl/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
print('''Set postgres url in the environment (or create a `.env` file in the notebooks folder).
Example: POSTGRES_BASE_URL="postgres://{user}:{password}@{host}:{port}/{dbname}"''')

Set postgres url in the environment (or create a `.env` file in the notebooks folder).
Example: POSTGRES_BASE_URL="postgres://{user}:{password}@{host}:{port}/{dbname}"


In [5]:
load_dotenv()
POSTGRES_BASE_URL = os.environ.get('POSTGRES_BASE_URL')

In [6]:
TABLENAME='tablename'
POSTGRES_URL = POSTGRES_BASE_URL + f'?tablename={TABLENAME}'

In [7]:
MODEL = 'Qwen/Qwen2.5-3B-Instruct'
#MODEL = 'openai/gpt-oss-20b'

In [8]:
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

'cuda'

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, device_map='auto')

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 526.20it/s, Materializing param=model.norm.weight]                              


In [10]:
lora_path = "grpo-lora/checkpoint-100"
model = PeftModel.from_pretrained(model, lora_path)

In [11]:
index = refactx.load_index(
    POSTGRES_URL, 
    #tokenizer,
    #configkey=-200,
    #cache='simple'
)

Applying index config...


In [12]:
index.get_config()

Applying index config...


{'switch_parameter': 7,
 'rootkey': -100,
 'tokenizer_name': 'Qwen/Qwen2.5-1.5B-Instruct'}

In [13]:
streamer = TextStreamer(tokenizer)

In [14]:
question = 'Is Johnny Depp older than Brad Pitt?'

prompted_texts = [refactx.apply_prompt_template(tokenizer, question=question, prompt_template=PROMPT_TEMPLATE)]

In [15]:
inputs = tokenizer(prompted_texts, return_tensors='pt', padding=True, padding_side='right')
inputs = inputs.to(model.device)
print(inputs['input_ids'].shape)

torch.Size([1, 176])


In [16]:
model.device

device(type='cuda', index=0)

In [17]:
# no need for num_beams=1
#refactx.patch_model(model)

In [18]:
num_beams = 1
num_batches = 1

auto_streamer = streamer if num_beams == 1 else None

In [19]:
constrained_processor = refactx.get_constrained_logits_processor(tokenizer, index, num_beams, num_batches)

In [20]:
logits_processor_list = constrained_processor

model.eval()
start = time.time()

with torch.no_grad():
    out = model.generate(
        **inputs,
        logits_processor=logits_processor_list,
        max_new_tokens=800,
        streamer = auto_streamer,
        do_sample = False,
        temperature = None,
        top_k=None,
        num_beams=num_beams,
        num_return_sequences=num_beams,
        use_cache=True,
        top_p=None,
        min_p=None,
    )

print('Elapsed', time.time() - start)

<|im_start|>system
You are a helpful question-answering assistant that bases its answers on facts from a knowledge base.

    You receive an input question.

    You determine the reasoning path needed to answer.

    You MUST get relevant facts with the "Fact:" command (e.g., "Fact: <Smith> <date of birth> <2000-10-01>"). You MUST rely on these facts and use them a proof for your answer.
    While getting facts you continue the reasoning explaining it step by step.

    You conclude with a concise answer that MUST be based on the proofs you found with "Fact:".

If you didn't find proofs with "Fact:" that support an answer you stop and you reply: "I don't know.".

<|im_end|>
<|im_start|>user
Is Johnny Depp older than Brad Pitt?<|im_end|>
<|im_start|>assistant
Answer:

Fact:
 <Brad Pitt> <date of birth> <1963-12-18T00:00:00Z> .<bradpitt>
 Fact:
 <Johnny Depp> <date of birth> <1963-06-09T00:00:00Z> .<johndepp>

Answer:

Answer:

Answer:

Answer:

Answer:

Answer:

Answer:

Answer:

Answe

KeyboardInterrupt: 

### Visualize ReFactX output

In [40]:
_from = len(inputs.input_ids[0]) # 0
for i in range(out.shape[0]):
    print('-'*30, sum(out[i][_from:]), len(out[i][_from:]))
    print(tokenizer.decode(out[i][_from:]))

------------------------------ tensor(10312752, device='cuda:0') 800
Answer: Yes

Explanation:

 Fact:
 <Brad Pitt> <date of birth> <1963-12-18T00:00:00Z> . 

 Answer: Yes

 Proof:

 Fact: <Brad Pitt> <occupation> <Actor> . 
 Answer: Actor

 Fact: <Brad Pitt> <place of birth> <Shawnee, Oklahoma> . 
 Answer: Shawnee, Oklahoma

 Fact: <Brad Pitt> <occupation> <Film director> . 
 Answer: Film director

 Fact: <Brad Pitt> <occupation> <Film producer> . 
 Answer: Film producer

 Fact: <Brad Pitt> <occupation> <Television producer> . 
 Answer: Television producer

 Fact: <Brad Pitt> <occupation> <Model (person)> . 
 Answer: Model (person)

 Fact: <Brad Pitt> <occupation> <Executive producer> . 
 Answer: Executive producer

 Fact: <Brad Pitt> <occupation> <voice actor (person who provides voice-overs for a character in films, animation, video games, or in other media)> . 
 Answer: Voice actor (person who provides voice-overs for a character in films, animation, video games, or in other media)

### Generated Facts

In [21]:
for i, triple in enumerate(refactx.get_constrained_states()[0][0].generated_triples):
    print(i, tokenizer.decode(triple), end='\n')

0  <Brad Pitt> <date of birth> <1963-12-18T00:00:00Z> .
1  <Johnny Depp> <date of birth> <1963-06-09T00:00:00Z> .
